Gather updated lists of files, samples, metadata\
KL 2 April 2026\
Set up a database, just with sample information

In [19]:
%reset -f
#%whos #also useful at times

In [20]:
import pandas as pd
import os
import pdb

#need this to see the full column width
pd.set_option('display.max_colwidth', None)

In [21]:
# #now I see why Ben was deleting the database...otherwise get multiple inserts
# but I cannot get this to work as it is still in use and I am having trouble closing it.
# %tried;
# session.close()
# engine.dispose()
# def delete_db():
#     print('Deleting database')
#     db_path = 'new_database.db'
#     if os.path.exists(db_path):
# #         os.unlink(db_path)
#         os.remove(db_path)

In [22]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [23]:
# define the classes

class DiscreteInfo(Base):
    __tablename__ = 'discrete'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cruise = Column(String)
    cast = Column(String)
    niskin = Column(String)
    nominalDepth = Column(String)
    V1V2data = Column(String)
    V4data = Column(String)
    
    #need this next row to get the nice output (other get a generic thing ?: <__main__.DiscreteInfo object at 0x000001A3FC7A0F70>)
    def __repr__(self):
        return f"<DiscreteInfo(bottleID='{self.bottleID}', cruise='{self.cruise}')>"


class SeqInfoV1V2(Base):
    __tablename__ = 'sequencingV1V2'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V1V2data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self):
        return f"<SeqInfoV1V2(bottleID='{self.bottleID}', filename='{self.filename}', V1V2data ='{self.V1V2data}')>"
    
class SeqInfoV4(Base):
    __tablename__ = 'sequencingV4'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V4data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, V4data={self.V4data!r})"

    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"index(id={self.id!r}, filename={self.filename!r})"

In [24]:
# create the database tables
Base.metadata.create_all(engine)

In [25]:
# # insert some data, setup functions, one per data type
def load_discrete_info():
    print('Loading discrete sample information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'BATS_BS_COMBINED_MASTER_mini.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name='DATA'))

    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = DiscreteInfo()
        db.bottleID = row['New_ID'] 
        db.cruise = row['Cruise_ID']
        db.cast = row['Cast']
        db.niskin = row['Niskin']
        db.nominalDepth = row['Nominal_Depth']
        session.add(db)
    
    session.commit()
    
def load_V4_sequencing_info():
    print('Loading V4 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = SeqInfoV4()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FilenameinCyverse']
        db.V4data = fName
        session.add(db)
    
    session.commit()
    
def load_V1V2_sequencing_info():
    print('Loading V1V2 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V1V2_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),
                                   dtype={'Bottle ID':str,'Cruise':str,'Cast':str,'Depth':str}))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !
    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = SeqInfoV1V2()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FileName']
        db.V1V2data = fName
        session.add(db)
    
    session.commit()

def load_cyverse_info():
    print('Loading sequencing information')
    dataDir = '../test_data/BIOS-SCOPE time series/'
    fName = 'files_shortList.txt'
    df = pd.read_csv(os.path.join(dataDir,fName),sep='\t',header=None,comment = '#')

    #strip off the end of the filename
    for index,row in df.iterrows():
        #file = os.path.basename(row.to_string()).strip('fastq.gz')
        file = os.path.basename(row.to_string()).strip('.gz')
        df.loc[index,'filename'] = file

    session = Session()
    for index, row in df.iterrows():
        db = CyverseInfo()
        db.filename = row['filename'] 
        session.add(db)
    
    session.commit()

In [26]:
#now run the functions
load_V4_sequencing_info()
load_V1V2_sequencing_info()
load_cyverse_info()
load_discrete_info()

Loading V4 sequencing information
Loading V1V2 sequencing information
Loading sequencing information
Loading discrete sample information


In [27]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'sequencingV1V2', 'sequencingV4']


In [28]:
from sqlalchemy import create_engine, inspect, MetaData, Table
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)
#print(f"Tables found: {metadata_obj.tables.keys()}") #

#mmm I think these lines put the data into a form I can use here but are NOT altering the existing database
# which is great, until I want to update the existing database
user_seqV4 = Table('sequencingV4', metadata_obj, autoload_with=engine)
user_seqV1V2 = Table('sequencingV1V2', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)

user_discrete

Table('discrete', MetaData(), Column('id', INTEGER(), table=<discrete>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<discrete>), Column('cruise', VARCHAR(), table=<discrete>), Column('cast', VARCHAR(), table=<discrete>), Column('niskin', VARCHAR(), table=<discrete>), Column('nominalDepth', VARCHAR(), table=<discrete>), Column('V1V2data', VARCHAR(), table=<discrete>), Column('V4data', VARCHAR(), table=<discrete>), schema=None)

In [29]:
#working on join_from ... keep this as an example

# # #now, with the discrete data, find the rows there with matching Bottle ID in the seqInfo file
# #set this up as a left outer join (all rows of discrete and only those rows of seqdata that match)
# #start tidying this up to make it useful. Plan is to ultimately send out one table with all
# #the discrete information and columns for cases where there is V1V2, V4, mtabs...

# stmt = select(
#     user_discrete.c.bottleID.label("New_ID"), 
#     user_discrete,
#     user_seqV4.c.V4data
# ).join_from(
#     user_discrete,
#     user_seqV4,
#     user_discrete.c.bottleID == user_seqV4.c.bottleID,
#     isouter=True
# )

# #session.scalars(stmt).one()
# with Session(engine) as session:
# #     for row in session.execute(stmt):
# #         print(row)
#     rows = session.execute(stmt).all()
#     table_data = [row._mapping for row in rows]
#     df = pd.DataFrame(table_data)

### new path from here...don't create a dataframe (yet)...update the existing database

In [30]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [31]:
#see all of what is in table
session.query(SeqInfoV4).all()

[User(id=1, name='1035501701', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=2, name='1035501703', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=3, name='1035501705', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=4, name='1035501707', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=5, name='1035501709', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=6, name='1035501711', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=7, name='1035501713', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=8, name='1035501715', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=9, name='1035501717', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=10, name='1035501719', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=11, name='1035501721', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=12, name='1035501723', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=13, name='1035601501', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=14, name='10

In [32]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV4.c.V4data)
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V4data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()    

In [33]:
#move on to V1V2 (surely there is a better way to do this...)

In [34]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [35]:
# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV1V2.c.V1V2data)
    .where(user_discrete.c.bottleID == user_seqV1V2.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V1V2data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

In [37]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    Column('V4data',String),
    Column('V1V2data',String)              
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)

df.to_csv('temp5.csv')

In [ ]:
#Stick some code below this spot as a holding zone

raise SystemExit("Stop execution here")

In [ ]:
#apparentely a new way to do this (though the old way is still supported)

In [466]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

In [467]:
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [468]:
Base.metadata

MetaData()

In [469]:
from typing import List
from typing import Optional
from sqlalchemy import Column, String
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class TestingNew(Base):
    __tablename__ = 'testingNew'
    id: Mapped[int] = mapped_column(primary_key=True)
    bottleID: Mapped[int] = mapped_column(String(10))
    cruise: Mapped[str] = mapped_column(String(10))
    cast: Mapped[int] = mapped_column(String(10))
    niskin: Mapped[int] = mapped_column(String(10))
    nominalDepth: Mapped[int] = mapped_column(String(10))
    
# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

In [ ]:
from sqlalchemy import select,insert

select_stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    user_seq.c.getDataHere
).join_from(
    user_discrete,
    user_seq,
    user_discrete.c.bottleID == user_seq.c.bottleID,
    isouter=True
)

insert_stmt = insert(user_discrete).from_select(
    ["id","getDataHere"],select_stmt)

# select_stmt = select(user_table.c.id, user_table.c.name + "@aol.com")
# insert_stmt = insert(address_table).from_select(
#     ["user_id", "email_address"], select_stmt
# )
print(select_stmt) #OK
print(insert_stmt) #fails: key error

In [ ]:
user_table_seqInfo.primary_key

In [ ]:
metadata_obj.create_all(engine)

In [ ]:
##now we want to declare our classes
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

#mote difference in syntax from example...this is new in SQLAlchemy 1.4

class SeqInfo(Base):
    __tablename__ = 'sequencingInfo'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, fullname={self.filename!r})"
    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.filename!r})"

In [ ]:
Base.metadata.create_all(engine)

In [ ]:
metadata_obj

In [ ]:
#now insert data

In [ ]:
from sqlalchemy.orm import sessionmaker, Session
from datetime import datetime, time
from tqdm import tqdm

In [ ]:
def load_sequencing_info():
    print('Loading sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db_si = SeqInfo()
        db_si.bottleID = row['BottleID'] 
        db_si.cast = row['Cast']
        #db_si.filename = row['FilenameInCyverse']
        session.add(db_si)
    session.commit()

In [ ]:
load_sequencing_info()

In [ ]:
#start with the sequence data, Luis gave me three lists. 
#Merge these and pull out relevant details, export to CSV file that will be read into the database
seqDir = '../test_data/Luis_fileLists'

dir_list = os.listdir(seqDir)
dir_list_full = [os.path.join(seqDir,f) for f in os.listdir(seqDir)] 